# M14 NUTS HMC en Kaggle T4 (regen cate_nuts.pkl + 4 parquets nuevos)

**Objetivo**: re-correr M14 (NUTS HMC 4 chains × 1000+1000) en GPU Kaggle (~7 min) para regenerar:
- `cate_nuts.pkl` (409 MB, gitignored, local-only)
- `posterior_probs.parquet` (NEW, ~25 KB)
- `pressure_player.parquet` (NEW, ~15 KB)
- `intra_corr_player.parquet` (NEW, ~15 KB)
- `scenarios_player.parquet` (NEW, ~100 KB)
- `diagnostics.parquet`, `posterior_player.parquet`, etc. (refresh)

**Antes de empezar (UI Kaggle)**:
1. Crear notebook nuevo
2. Settings (panel derecho):
   - **Accelerator**: GPU T4 x1 (no TPU, no GPU P100)
   - **Persistence**: Files only
   - **Internet**: On
3. Subir este `.ipynb` o pegar las celdas en orden

Repo publico: clone HTTPS directo sin token. Ejecucion lineal 1→2→3→4→5→6. Total ~10 min.

## Celda 1 — Clone del repo (publico)

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/jaime-oriol/PCJ.git"
WORK = "/kaggle/working/PCJ"

if os.path.exists(WORK):
    subprocess.run(["rm", "-rf", WORK], check=True)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
os.chdir(WORK)
print("repo en:", os.getcwd())
print("branch:", subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip())
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip())

## Celda 2 — Instalar deps (numpyro + jax cuda12 + polars)

Kaggle T4 viene con CUDA 12.x. Forzamos versiones compatibles.

In [ ]:
!pip install -q --upgrade "jax[cuda12]==0.4.38" "numpyro==0.16.1" "polars==1.39.3" pyarrow

# CRITICO: polars 1.39.3 (no 1.18) — los parquets cacheados se escribieron con 1.39
# y polars 1.18 falla al leer string cols con `invalid from_physical(String) for Binary`

## Celda 3 — Verificar GPU JAX

In [ ]:
import jax, numpyro
print("jax", jax.__version__, "backend:", jax.default_backend())
print("devices:", jax.devices())
print("numpyro", numpyro.__version__)
assert jax.default_backend() == "gpu", "GPU no detectada: revisar Settings -> Accelerator T4 x1"

## Celda 4 — Correr M14 (NUTS HMC 4 chains × 1000+1000)

Dump tambien los 4 parquets nuevos (`_dump_all_sample_derivatives`).

Tiempo esperado T4: ~7 min.

In [ ]:
import sys, time, pickle
from pathlib import Path
sys.path.insert(0, "/kaggle/working/PCJ/src")
import numpyro
numpyro.set_host_device_count(4)

from M14_cate import (compute_all, build_delta_panel, attach_pff_grades,
                      compute_indices, compute_rankings,
                      _dump_all_sample_derivatives)

DERIVED = Path("/kaggle/working/PCJ/data/parquet/derived/cate")
PKL = DERIVED / "model" / "cate_nuts.pkl"

# Resume mode: si ya existe cate_nuts.pkl (NUTS hecho), salta el HMC y solo
# completa los pasos faltantes [6]+[7]. Si no existe, corre compute_all entero.
if PKL.exists():
    print(f"=== RESUME: cate_nuts.pkl encontrado ({PKL.stat().st_size/1024/1024:.0f} MB) ===")
    with open(PKL, "rb") as f:
        fit = pickle.load(f)
    print(f"fit cargado, samples shape eta_ga: {fit['samples']['eta_ga'].shape}")

    panel = build_delta_panel(cache=True)
    panel = attach_pff_grades(panel)
    print(f"panel: {panel.shape}")

    out_paths = {
        "indices":         DERIVED / "indices.parquet",
        "rankings":        DERIVED / "rankings.parquet",
        "posterior_probs": DERIVED / "posterior_probs.parquet",
        "pressure_player": DERIVED / "pressure_player.parquet",
        "intra_corr":      DERIVED / "intra_corr_player.parquet",
        "scenarios":       DERIVED / "scenarios_player.parquet",
    }

    print("[6] Indices + rankings...")
    t0 = time.time()
    idx = compute_indices(fit)
    rank = compute_rankings(idx, panel)
    idx.write_parquet(out_paths["indices"], compression="snappy")
    rank.write_parquet(out_paths["rankings"], compression="snappy")
    print(f"   {time.time()-t0:.1f}s")

    print("[7] Dump samples derivatives...")
    t0 = time.time()
    _dump_all_sample_derivatives(fit, out_paths)
    print(f"   {time.time()-t0:.1f}s")
    print("\n=== RESUME DONE ===")
else:
    print("=== FRESH RUN: no pkl, corriendo compute_all entero (~17 min NUTS) ===")
    t0 = time.time()
    paths = compute_all(cache=True, overwrite=True)
    print(f"\n=== M14 DONE en {(time.time()-t0)/60:.1f} min ===")
    for k, p in paths.items():
        sz = p.stat().st_size if p.exists() else 0
        print(f"  {k:20s} {sz/1024:>8.1f} KB  {p}")

## Celda 5 — Verificar diagnostics + nuevos parquets

In [ ]:
import polars as pl
from pathlib import Path
CATE = Path("/kaggle/working/PCJ/data/parquet/derived/cate")

diag = pl.read_parquet(CATE / "diagnostics.parquet")
n_diverged = diag.filter(~pl.col("converged")).height
print(f"Diagnostics: {diag.height} params, {n_diverged} no convergidos (R-hat>=1.05 o ESS<400)")

for name in ["posterior_probs", "pressure_player", "intra_corr_player", "scenarios_player"]:
    df = pl.read_parquet(CATE / f"{name}.parquet")
    print(f"  {name}.parquet: {df.shape}")

## Celda 6 — Empaquetar artefactos para descarga

Genera `/kaggle/working/m14_artifacts.zip`. Click derecho > Download desde el panel **Output** del notebook (no `Working Directory`).

In [ ]:
import shutil, os
from pathlib import Path
SRC = Path("/kaggle/working/PCJ/data/parquet/derived/cate")
DST = Path("/kaggle/working/m14_artifacts")
if DST.exists():
    shutil.rmtree(DST)
DST.mkdir()
(DST / "model").mkdir()

# 4 parquets nuevos + diagnostics/posterior/indices/rankings/ppc/corr refresh + pkl
for f in ["posterior_probs.parquet", "pressure_player.parquet",
          "intra_corr_player.parquet", "scenarios_player.parquet",
          "posterior_player.parquet", "indices.parquet", "rankings.parquet",
          "posterior_corr.parquet", "diagnostics.parquet", "ppc.parquet"]:
    shutil.copy(SRC / f, DST / f)
shutil.copy(SRC / "model" / "cate_nuts.pkl", DST / "model" / "cate_nuts.pkl")

out = shutil.make_archive("/kaggle/working/m14_artifacts", "zip", DST)
print("zip:", out, f"({os.path.getsize(out)/1024/1024:.1f} MB)")
print("\nDescarga desde: panel Output -> m14_artifacts.zip")